# Whisper ASR + LLM — Colab to local server

## 1. Install dependencies

In [ ]:
!python -m pip install bitsandbytes --prefer-binary --extra-index-url=https://jllllll.github.io/bitsandbytes-windows-webui

!pip install --upgrade pip
!pip install transformers accelerate
!pip install peft==0.10.0 trl==0.8.6
!pip install -U gradio
!pip install evaluate
!pip install soundfile
!pip install librosa

!pip install git+https://github.com/openai/whisper.git -q

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths & config

In [ ]:
import os, sys

DRIVE_ROOT       = '/content/drive/MyDrive'
BASE_DIR         = f'{DRIVE_ROOT}/capstone_design'
SRC_DIR          = f'{BASE_DIR}/whisper_src'
WHISPER_BASE     = f'{BASE_DIR}/whisper-merged2'
WHISPER_ADAPTER  = f'{BASE_DIR}/whisper-dialect-turbo-qlora/checkpoint-1880'
LLM_MERGED       = f'{BASE_DIR}/llama3.2-merged'
LOCAL_SERVER_URL = 'https://8db3-221-161-190-89.ngrok-free.app/test'

assert os.path.isfile(f'{SRC_DIR}/voice_command_pipeline.py'), \
    f'Missing: {SRC_DIR}/voice_command_pipeline.py'
for p in [WHISPER_BASE, WHISPER_ADAPTER, LLM_MERGED]:
    assert os.path.isdir(p), f'Missing: {p}'

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print('Paths OK.')
print('server  :', LOCAL_SERVER_URL)

## 4. Load Whisper (base + QLoRA adapter)

In [ ]:
import torch
from peft import PeftModel
from transformers import (AutoTokenizer, BitsAndBytesConfig,
                          WhisperForConditionalGeneration, WhisperProcessor)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

base_model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_BASE,
    quantization_config=quantization_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(base_model, WHISPER_ADAPTER)
model.eval()

processor = WhisperProcessor.from_pretrained(WHISPER_BASE)
whisper_tokenizer = AutoTokenizer.from_pretrained(WHISPER_BASE)

print('Whisper ready on', next(model.parameters()).device)

## 5. Load LLM (merged, no adapter)

In [ ]:
from transformers import AutoModelForCausalLM

# T4(Turing)는 bf16 네이티브 지원이 없어 fp16이 빠르다
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MERGED,
    torch_dtype=DTYPE,
    device_map='auto',
)
llm_model.eval()

tokenizer = AutoTokenizer.from_pretrained(LLM_MERGED)
llm_model.config.pad_token_id = tokenizer.pad_token_id

print('LLM ready on', next(llm_model.parameters()).device, '|', DTYPE)

## 6. Push to Hugging Face

### 6-1. Login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN        = userdata.get('HF_TOKEN')
HF_REPO_WHISPER = 'minsu0567/Capstone-Design-Whisper-Dialect'
HF_REPO_LLM     = 'minsu0567/Capstone-Design-Llama3.2-Command'

login(token=HF_TOKEN)
print('HF login OK.')

### 6-2. Whisper — merge QLoRA adapter, then push

In [ ]:
import gc

merge_base = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_BASE,
    torch_dtype=torch.float16,
    device_map='cpu',
)
merged = PeftModel.from_pretrained(merge_base, WHISPER_ADAPTER, torch_dtype=torch.float16)
merged = merged.merge_and_unload()
merged.generation_config.language = 'korean'
merged.generation_config.task = 'transcribe'

merged.push_to_hub(HF_REPO_WHISPER, token=HF_TOKEN)
processor.push_to_hub(HF_REPO_WHISPER, token=HF_TOKEN)
whisper_tokenizer.push_to_hub(HF_REPO_WHISPER, token=HF_TOKEN)
print('Pushed merged Whisper ->', HF_REPO_WHISPER)

del merge_base, merged
gc.collect()
torch.cuda.empty_cache()

### 6-3. Llama 3.2 — push merged model

In [ ]:
llm_model.push_to_hub(HF_REPO_LLM, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_LLM, token=HF_TOKEN)
print('Pushed LLM ->', HF_REPO_LLM)

## 7. Launch Gradio

In [ ]:
from voice_command_pipeline import build_interface

interface = build_interface(
    whisper_model     = model,
    processor         = processor,
    whisper_tokenizer = whisper_tokenizer,
    llm_model         = llm_model,
    llm_tokenizer     = tokenizer,
    server_url        = LOCAL_SERVER_URL,
)
interface.launch(share=True)